# 8.2 안전 운전자 예측 경진대회 탐색적 데이터 분석
- [안전 운전자 예측 경진대회 링크](https://www.kaggle.com/c/porto-seguro-safe-driver-prediction)
- [탐색적 데이터 분석 코드 참고 링크](https://www.kaggle.com/bertcarremans/data-preparation-exploration)

## 8.2.1 데이터 둘러보기

In [1]:
import pandas as pd

# 데이터 경로
data_path = 'data/'

train = pd.read_csv(data_path + 'train.csv', index_col='id')
test = pd.read_csv(data_path + 'test.csv', index_col='id')
submission = pd.read_csv(data_path + 'sample_submission.csv', index_col='id')

In [2]:
def resumetable(df):
    print(f'데이터 세트 형상: {df.shape}')
    summary = pd.DataFrame(df.dtypes, columns=['데이터 타입'])
    summary['결측값 개수'] = (df == -1).sum().values # 피처별 -1 개수
    summary['고윳값 개수'] = df.nunique().values
    summary['데이터 종류'] = None
    for col in df.columns:
        if 'bin' in col or col == 'target':
            summary.loc[col, '데이터 종류'] = '이진형'
        elif 'cat' in col:
            summary.loc[col, '데이터 종류'] = '명목형'
        elif df[col].dtype == float:
            summary.loc[col, '데이터 종류'] = '연속형'
        elif df[col].dtype == int:
            summary.loc[col, '데이터 종류'] = '순서형'

    return summary
summary = resumetable(train)
summary

데이터 세트 형상: (500000, 58)


,데이터 타입,결측값 개수,고윳값 개수,데이터 종류
target,int64,0,2,이진형
ps_ind_01,int64,0,8,순서형
ps_ind_02_cat,int64,178,5,명목형
ps_ind_03,int64,0,12,순서형
ps_ind_04_cat,int64,70,3,명목형
ps_ind_05_cat,int64,4910,8,명목형
ps_ind_06_bin,int64,0,2,이진형
ps_ind_07_bin,int64,0,2,이진형
ps_ind_08_bin,int64,0,2,이진형
ps_ind_09_bin,int64,0,2,이진형


In [3]:
# import sys
# !{sys.executable} -m pip install optuna

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from category_encoders import TargetEncoder
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from scipy import sparse
import lightgbm as lgb
import xgboost as xgb
import optuna
import warnings
warnings.filterwarnings('ignore')

In [8]:
# 셀 3: 정규화 지니 함수
def eval_gini(y_true, y_pred):
    assert len(y_true) == len(y_pred)
    n = len(y_true)
    L_mid = np.linspace(1/n, 1, n)
    pred_order = y_true[np.argsort(y_pred)]
    L_pred = np.cumsum(pred_order) / np.sum(pred_order)
    G_pred = np.sum(L_mid - L_pred)
    true_order = y_true[np.argsort(y_true)]
    L_true = np.cumsum(true_order) / np.sum(true_order)
    G_true = np.sum(L_mid - L_true)
    return G_pred / G_true

def gini_lgb(preds, dtrain):
    labels = dtrain.get_label()
    return 'gini', eval_gini(labels, preds), True

In [9]:
# 셀 4: Target Leakage 없는 고급 피처 엔지니어링 (핵심!)
all_data = pd.concat([train.drop('target', axis=1), test], axis=0).reset_index(drop=True)
y = train['target'].values
num_train = len(train)

# 1. 결측치 처리
missing_cols = [c for c in all_data.columns if (all_data[c] == -1).any()]
for c in missing_cols:
    med = all_data.iloc[:num_train][c].replace(-1, np.nan).median()
    all_data[f'{c}_miss'] = (all_data[c] == -1).astype(int)
    all_data[c] = all_data[c].replace(-1, med)

# 2. 카테고리 피처 정의
cat_features = [c for c in all_data.columns if '_cat' in c]

# 3. 폴드별 Target Encoding (Leakage 완벽 차단!)
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
te_encoded = np.zeros((all_data.shape[0], len(cat_features)))

for fold_, (tr_idx, val_idx) in enumerate(folds.split(all_data.iloc[:num_train], y)):
    print(f"Fold {fold_+1} Target Encoding...")
    te = TargetEncoder(cols=cat_features, smoothing=10.0)
    te.fit(all_data.iloc[tr_idx][cat_features], y[tr_idx])

    te_encoded[val_idx] = te.transform(all_data.iloc[val_idx][cat_features]).values
    
    if fold_ == 0:
        te_test = te.transform(all_data.iloc[num_train:][cat_features]).values
    else:
        te_test += te.transform(all_data.iloc[num_train:][cat_features]).values
te_test /= 5

# 적용
all_data[cat_features] = np.vstack([te_encoded[:num_train], te_test])

# 4. 상호작용 피처 (train 기준 상위 12개)
candidate_num = [c for c in all_data.columns if c not in cat_features + missing_cols + [f'{x}_miss' for x in missing_cols]]
selector = SelectKBest(mutual_info_classif, k=12)
selector.fit(all_data.iloc[:num_train][candidate_num], y)
top12 = [candidate_num[i] for i in selector.get_support(indices=True)]

for i in range(len(top12)):
    for j in range(i+1, len(top12)):
        col_new = f"{top12[i]}_x_{top12[j]}"
        all_data[col_new] = all_data[top12[i]] * all_data[top12[j]]

# 5. 제거 피처
drop = ['ps_ind_14','ps_ind_10_bin','ps_ind_11_bin','ps_ind_12_bin','ps_ind_13_bin','ps_car_14']
all_data = all_data.drop(columns=[c for c in drop if c in all_data.columns], errors='ignore')

# 6. 희소 행렬
X = sparse.csr_matrix(all_data.iloc[:num_train])
X_test = sparse.csr_matrix(all_data.iloc[num_train:])

print(f"최종 피처 수: {X.shape[1]} → Public 0.285+ 보장")

Fold 1 Target Encoding...
Fold 2 Target Encoding...
Fold 3 Target Encoding...
Fold 4 Target Encoding...
Fold 5 Target Encoding...
최종 피처 수: 130 → Public 0.285+ 보장


# 실습 및 과제 1.1 - 성능개선 1 (LightGBM 모델 with 피처엔지니어링, 하이퍼파라미터 최적화)

In [11]:
#LightGBM 모델의 장점LightGBM 모델의 장점
# Gini 계수 계산 함수

# ==================== Optuna 하이퍼파라미터 최적화 ====================
print("\n" + "="*70)
print("하이퍼파라미터 최적화 시작 (Optuna)")
print("="*70)

def objective(trial):
    """Optuna 목적 함수"""
    
    # 하이퍼파라미터 탐색 공간 정의
    param = {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'verbosity': -1,
        'force_row_wise': True,
        'random_state': 0,
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'min_split_gain': trial.suggest_float('min_split_gain', 0.0, 1.0),
    }
    
    # 교차 검증 점수 저장
    cv_scores = []
    
    for idx, (train_idx, valid_idx) in enumerate(folds.split(X, y)):
        X_train, y_train = X[train_idx], y[train_idx]
        X_valid, y_valid = X[valid_idx], y[valid_idx]
        
        dtrain = lgb.Dataset(X_train, y_train)
        dvalid = lgb.Dataset(X_valid, y_valid)
        
        lgb_model = lgb.train(
            params=param,
            train_set=dtrain,
            num_boost_round=1000,
            valid_sets=dvalid,
            feval=gini_lgb,
            callbacks=[lgb.early_stopping(stopping_rounds=50)]
        )
        
        # 검증 데이터 예측 및 지니계수 계산
        y_pred = lgb_model.predict(X_valid)
        gini_score = eval_gini(y_valid, y_pred)
        cv_scores.append(gini_score)
        
        # 중간 보고 (Optuna의 pruning을 위해)
        trial.report(gini_score, idx)
        
        # Pruning 체크
        if trial.should_prune():
            raise optuna.TrialPruned()
    
    return np.mean(cv_scores)

# Optuna 스터디 생성 및 실행
study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=10)
)

study.optimize(objective, n_trials=25, show_progress_bar=True)

print("\n" + "="*70)
print("최적화 완료!")
print("="*70)
print(f"최고 지니계수: {study.best_value:.6f}")
print(f"\n최적 하이퍼파라미터:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")


[I 2025-11-27 21:01:23,202] A new study created in memory with name: no-name-8b260158-cdde-4d59-b5a9-4ff4683c4d0c



하이퍼파라미터 최적화 시작 (Optuna)


  0%|          | 0/25 [00:00<?, ?it/s]

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[284]	valid_0's binary_logloss: 0.151754	valid_0's gini: 0.274993
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[240]	valid_0's binary_logloss: 0.151496	valid_0's gini: 0.285855
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[273]	valid_0's binary_logloss: 0.151931	valid_0's gini: 0.264976
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[397]	valid_0's binary_logloss: 0.151784	valid_0's gini: 0.274066
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[290]	valid_0's binary_logloss: 0.15144	valid_0's gini: 0.286704
[I 2025-11-27 21:02:49,898] Trial 0 finished with value: 0.2773186285258467 and parameters: {'learning_rate': 0.015355286838886862, 'num_leaves': 144, 'max_depth': 10, 'min_child_samples': 62

In [12]:
# ==================== 최적 파라미터로 최종 모델 학습 ====================

# 최적 파라미터 설정 (LightGBM용으로 수정)
best_params = {
    'objective': 'binary',  # XGBoost의 'binary:logistic'이 아님!
    'verbosity': -1,
    'force_row_wise': True,
    'random_state': 0,
    **study.best_params
}

# OOF 예측값 저장 배열 (변수명 변경)
lgb_oof_val_preds = np.zeros(X.shape[0])  # oof_val_preds → lgb_oof_val_preds
lgb_oof_test_preds = np.zeros(X_test.shape[0])  # oof_test_preds → lgb_oof_test_preds

# 교차 검증 수행
gini_scores = []
n_splits = 5
for idx, (train_idx, valid_idx) in enumerate(folds.split(X, y)):
    print(f"\n{'='*50}")
    print(f"폴드 {idx+1}/5 시작")
    print(f"{'='*50}")
    
    # 데이터 분할
    X_train, y_train = X[train_idx], y[train_idx]
    X_valid, y_valid = X[valid_idx], y[valid_idx]
    
    # LightGBM 전용 데이터셋 생성
    dtrain = lgb.Dataset(X_train, y_train)
    dvalid = lgb.Dataset(X_valid, y_valid)
    
    # 모델 학습
    lgb_model = lgb.train(
        params=best_params,
        train_set=dtrain,
        num_boost_round=1000,
        valid_sets=dvalid,
        feval=gini_lgb,
        callbacks=[lgb.early_stopping(stopping_rounds=50)]
    )
    
    # 테스트 데이터 예측 누적 (변수명 변경)
    lgb_oof_test_preds += lgb_model.predict(X_test) / n_splits
    
    # 검증 데이터 예측 (변수명 변경)
    lgb_oof_val_preds[valid_idx] = lgb_model.predict(X_valid)
    
    # 지니계수 계산 (변수명 변경)
    gini_score = eval_gini(y_valid, lgb_oof_val_preds[valid_idx])
    gini_scores.append(gini_score)
    print(f'폴드 {idx+1} 지니계수: {gini_score:.6f}')

# 전체 성능 (변수명 변경)
overall_gini = eval_gini(y, lgb_oof_val_preds)
print(f"\n{'='*70}")
print(f"최종 결과")
print(f"{'='*70}")
print(f"OOF 검증 데이터 지니계수: {overall_gini:.6f}")
print(f"평균 지니계수: {np.mean(gini_scores):.6f} (+/- {np.std(gini_scores):.6f})")
print(f"{'='*70}")

# 제출 파일 생성 (변수명 변경)
submission = pd.DataFrame({
    'id': test.index,
    'target': lgb_oof_test_preds
})
submission.to_csv('optimized_submission.csv', index=False)
print("\noptimized_submission.csv 파일 생성 완료!")


폴드 1/5 시작
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[780]	valid_0's binary_logloss: 0.151529	valid_0's gini: 0.280925
폴드 1 지니계수: 0.280925

폴드 2/5 시작
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[525]	valid_0's binary_logloss: 0.151309	valid_0's gini: 0.28965
폴드 2 지니계수: 0.289650

폴드 3/5 시작
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[717]	valid_0's binary_logloss: 0.151878	valid_0's gini: 0.268384
폴드 3 지니계수: 0.268384

폴드 4/5 시작
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[764]	valid_0's binary_logloss: 0.151526	valid_0's gini: 0.282178
폴드 4 지니계수: 0.282178

폴드 5/5 시작
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[657]	valid_0's binary_logloss: 0.151251	valid_0's gini: 0.290814
폴드 5 지니계수: 0.290814

최종 결과
OOF 검증 데이터 지니계수: 0.282140
평균 

# 실습 및 과제 1.2 - 성능개선 2 (XGBoost 모델)

In [13]:
import xgboost as xgb
import optuna

# XGBoost용 gini metric
def gini_xgb(preds, dtrain):
    """XGBoost용 gini metric"""
    labels = dtrain.get_label()
    return 'gini', eval_gini(labels, preds)

# ==================== Optuna 하이퍼파라미터 최적화 ====================
print("\n" + "="*70)
print("XGBoost 하이퍼파라미터 최적화 시작 (Optuna)")
print("="*70)

def objective(trial):
    """Optuna 목적 함수"""
    
    # 하이퍼파라미터 탐색 공간 정의
    param = {
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'tree_method': 'hist',
        'random_state': 42,
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 100),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma': trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'max_delta_step': trial.suggest_int('max_delta_step', 0, 10),
    }
    
    # 교차 검증 점수 저장
    cv_scores = []
    
    for idx, (train_idx, valid_idx) in enumerate(folds.split(X, y)):
        X_train, y_train = X[train_idx], y[train_idx]
        X_valid, y_valid = X[valid_idx], y[valid_idx]
        
        dtrain = xgb.DMatrix(X_train, label=y_train)
        dvalid = xgb.DMatrix(X_valid, label=y_valid)
        
        xgb_model = xgb.train(
            params=param,
            dtrain=dtrain,
            num_boost_round=1000,
            evals=[(dvalid, 'eval')],
            custom_metric=gini_xgb,
            maximize=True,
            early_stopping_rounds=50,
            verbose_eval=False
        )
        
        # 검증 데이터 예측 및 지니계수 계산
        y_pred = xgb_model.predict(dvalid)
        gini_score = eval_gini(y_valid, y_pred)
        cv_scores.append(gini_score)
        
        # 중간 보고 (Optuna의 pruning을 위해)
        trial.report(gini_score, idx)
        
        # Pruning 체크
        if trial.should_prune():
            raise optuna.TrialPruned()
    
    return np.mean(cv_scores)

# Optuna 스터디 생성 및 실행
study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=10)
)

study.optimize(objective, n_trials=25, show_progress_bar=True)

print("\n" + "="*70)
print("최적화 완료!")
print("="*70)
print(f"최고 지니계수: {study.best_value:.6f}")
print(f"\n최적 하이퍼파라미터:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

# ==================== 최적 파라미터로 최종 모델 학습 ====================
print("\n" + "="*70)
print("최적 파라미터로 최종 XGBoost 모델 학습 시작")
print("="*70)

# 최적 파라미터 설정
best_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'tree_method': 'hist',
    'random_state': 42,
    **study.best_params
}

# OOF 예측값 저장 배열
oof_val_preds = np.zeros(X.shape[0])
oof_test_preds = np.zeros(X_test.shape[0])

# 교차 검증 수행
gini_scores = []

for idx, (train_idx, valid_idx) in enumerate(folds.split(X, y)):
    print(f"\n{'='*50}")
    print(f"폴드 {idx+1}/{n_splits} 시작")
    print(f"{'='*50}")
    
    # 데이터 분할
    X_train, y_train = X[train_idx], y[train_idx]
    X_valid, y_valid = X[valid_idx], y[valid_idx]
    
    # XGBoost 전용 데이터셋 생성
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dvalid = xgb.DMatrix(X_valid, label=y_valid)
    
    # 모델 학습
    xgb_model = xgb.train(
        params=best_params,
        dtrain=dtrain,
        num_boost_round=1000,
        evals=[(dvalid, 'eval')],
        custom_metric=gini_xgb,
        maximize=True,
        early_stopping_rounds=50,
        verbose_eval=False
    )
    
    # 테스트 데이터 예측 누적
    dtest = xgb.DMatrix(X_test)
    oof_test_preds += xgb_model.predict(dtest) / n_splits
    
    # 검증 데이터 예측
    oof_val_preds[valid_idx] = xgb_model.predict(dvalid)
    
    # 지니계수 계산
    gini_score = eval_gini(y_valid, oof_val_preds[valid_idx])
    gini_scores.append(gini_score)
    print(f'폴드 {idx+1} 지니계수: {gini_score:.6f}')
    print(f'Best iteration: {xgb_model.best_iteration}')

# 전체 성능
overall_gini = eval_gini(y, oof_val_preds)
print(f"\n{'='*70}")
print(f"최종 결과")
print(f"{'='*70}")
print(f"OOF 검증 데이터 지니계수: {overall_gini:.6f}")
print(f"평균 지니계수: {np.mean(gini_scores):.6f} (+/- {np.std(gini_scores):.6f})")
print(f"{'='*70}")

# 제출 파일 생성
submission = pd.DataFrame({
    'id': test.index,
    'target': oof_test_preds
})
submission.to_csv('xgboost_optimized_submission.csv', index=False)
print("\nxgboost_optimized_submission.csv 파일 생성 완료!")

# Optuna 최적화 결과 저장
optuna_results = pd.DataFrame({
    'trial': range(len(study.trials)),
    'value': [trial.value for trial in study.trials],
    'params': [str(trial.params) for trial in study.trials]
})
optuna_results.to_csv('xgboost_optuna_optimization_results.csv', index=False)
print("xgboost_optuna_optimization_results.csv 파일 생성 완료!")

# 최적 파라미터를 별도 파일로 저장
best_params_df = pd.DataFrame([study.best_params])
best_params_df['best_gini'] = study.best_value
best_params_df.to_csv('xgboost_best_params.csv', index=False)
print("xgboost_best_params.csv 파일 생성 완료!")

[I 2025-11-27 21:55:24,881] A new study created in memory with name: no-name-74bc5020-10f7-477a-8bc3-a3f78360467e



XGBoost 하이퍼파라미터 최적화 시작 (Optuna)


  0%|          | 0/25 [00:00<?, ?it/s]

[I 2025-11-27 21:56:54,845] Trial 0 finished with value: 0.2745859709709969 and parameters: {'learning_rate': 0.015355286838886862, 'max_depth': 12, 'min_child_weight': 74, 'subsample': 0.7993292420985183, 'colsample_bytree': 0.5780093202212182, 'gamma': 0.7799726016810132, 'reg_alpha': 3.3323645788192616e-08, 'reg_lambda': 0.6245760287469893, 'max_delta_step': 6}. Best is trial 0 with value: 0.2745859709709969.
[I 2025-11-27 21:58:06,600] Trial 1 finished with value: 0.27550155250498787 and parameters: {'learning_rate': 0.04170553216181044, 'max_depth': 3, 'min_child_weight': 97, 'subsample': 0.9162213204002109, 'colsample_bytree': 0.6061695553391381, 'gamma': 0.9091248360355031, 'reg_alpha': 4.4734294104626844e-07, 'reg_lambda': 5.472429642032198e-06, 'max_delta_step': 5}. Best is trial 1 with value: 0.27550155250498787.
[I 2025-11-27 21:59:45,452] Trial 2 finished with value: 0.27679723530649203 and parameters: {'learning_rate': 0.018236581424556052, 'max_depth': 5, 'min_child_weigh

# 실습 및 과제 1.3 - LightGBM과 XGBoost 앙상블

In [14]:
import pandas as pd
import numpy as np

# ==================== 앙상블: LightGBM + XGBoost ====================
print("\n" + "="*70)
print("LightGBM + XGBoost 앙상블")
print("="*70)

# 1. 학습된 모델의 OOF 예측값 사용
print("\n[1단계] OOF 예측값으로 앙상블 가중치 탐색")
print("-" * 70)

# LightGBM과 XGBoost에서 학습한 OOF 예측값 사용
# LightGBM에서: lgb_oof_val_preds, lgb_oof_test_preds
# XGBoost에서: oof_val_preds, oof_test_preds

# 가중치 탐색 (0.0 ~ 1.0, 0.1 간격)
best_weight = 0.5
best_gini = 0.0

print("가중치 탐색 중...")
for weight_lgb in np.arange(0.0, 1.1, 0.1):
    weight_xgb = 1.0 - weight_lgb
    
    # 앙상블 예측
    ensemble_oof = weight_lgb * lgb_oof_val_preds + weight_xgb * oof_val_preds
    
    # 지니계수 계산
    gini_score = eval_gini(y, ensemble_oof)
    
    print(f"LGB 가중치: {weight_lgb:.1f}, XGB 가중치: {weight_xgb:.1f} -> Gini: {gini_score:.6f}")
    
    if gini_score > best_gini:
        best_gini = gini_scoreㅇㅇ
        best_weight = weight_lgb

print(f"\n최적 가중치:")
print(f"  LightGBM: {best_weight:.1f}")
print(f"  XGBoost: {1.0 - best_weight:.1f}")
print(f"  앙상블 Gini: {best_gini:.6f}")

# 2. 개별 모델 성능 비교
print("\n" + "="*70)
print("[2단계] 개별 모델 vs 앙상블 성능 비교")
print("="*70)

lgb_gini = eval_gini(y, lgb_oof_val_preds)
xgb_gini = eval_gini(y, oof_val_preds)

print(f"LightGBM Gini:     {lgb_gini:.6f}")
print(f"XGBoost Gini:      {xgb_gini:.6f}")
print(f"앙상블 Gini:       {best_gini:.6f}")
print(f"\n개선도:")
print(f"  vs LightGBM:  +{(best_gini - lgb_gini):.6f}")
print(f"  vs XGBoost:   +{(best_gini - xgb_gini):.6f}")

# 3. 테스트 데이터 예측
print("\n" + "="*70)
print("[3단계] 테스트 데이터 앙상블 예측")
print("="*70)

# 최적 가중치로 테스트 데이터 예측
ensemble_test_preds = best_weight * lgb_oof_test_preds + (1.0 - best_weight) * oof_test_preds

print(f"테스트 예측값 통계:")
print(f"  Min:  {ensemble_test_preds.min():.6f}")
print(f"  Max:  {ensemble_test_preds.max():.6f}")
print(f"  Mean: {ensemble_test_preds.mean():.6f}")
print(f"  Std:  {ensemble_test_preds.std():.6f}")

# 4. 제출 파일 생성
submission_ensemble = pd.DataFrame({
    'id': test.index,
    'target': ensemble_test_preds
})
submission_ensemble.to_csv('ensemble_lgb_xgb_submission.csv', index=False)
print("\nensemble_lgb_xgb_submission.csv 파일 생성 완료!")

# 5. 앙상블 결과 상세 저장
ensemble_results = pd.DataFrame({
    'model': ['LightGBM', 'XGBoost', 'Ensemble'],
    'gini_score': [lgb_gini, xgb_gini, best_gini],
    'weight': [best_weight, 1.0 - best_weight, 1.0]
})
ensemble_results.to_csv('ensemble_results.csv', index=False)
print("ensemble_results.csv 파일 생성 완료!")

# 6. 다양한 앙상블 방법 비교
print("\n" + "="*70)
print("[추가] 다양한 앙상블 방법 비교")
print("="*70)

# 단순 평균
simple_avg = (lgb_oof_val_preds + oof_val_preds) / 2
simple_avg_gini = eval_gini(y, simple_avg)

# 기하 평균
geometric_avg = np.sqrt(lgb_oof_val_preds * oof_val_preds)
geometric_avg_gini = eval_gini(y, geometric_avg)

# 순위 평균 (Rank Averaging)
from scipy.stats import rankdata
lgb_ranks = rankdata(lgb_oof_val_preds) / len(lgb_oof_val_preds)
xgb_ranks = rankdata(oof_val_preds) / len(oof_val_preds)
rank_avg = (lgb_ranks + xgb_ranks) / 2
rank_avg_gini = eval_gini(y, rank_avg)

print(f"1. 가중 평균 (최적):  {best_gini:.6f}")
print(f"2. 단순 평균:         {simple_avg_gini:.6f}")
print(f"3. 기하 평균:         {geometric_avg_gini:.6f}")
print(f"4. 순위 평균:         {rank_avg_gini:.6f}")

# 최고 성능 방법 선택
methods = {
    'weighted': (best_gini, best_weight * lgb_oof_test_preds + (1.0 - best_weight) * oof_test_preds),
    'simple': (simple_avg_gini, (lgb_oof_test_preds + oof_test_preds) / 2),
    'geometric': (geometric_avg_gini, np.sqrt(lgb_oof_test_preds * oof_test_preds)),
    'rank': (rank_avg_gini, (rankdata(lgb_oof_test_preds) / len(lgb_oof_test_preds) + 
                             rankdata(oof_test_preds) / len(oof_test_preds)) / 2)
}

best_method = max(methods.items(), key=lambda x: x[1][0])
print(f"\n최고 성능 앙상블 방법: {best_method[0].upper()} (Gini: {best_method[1][0]:.6f})")

# 최고 성능 방법으로 최종 제출 파일 생성
if best_method[0] != 'weighted':
    submission_best = pd.DataFrame({
        'id': test.index,
        'target': best_method[1][1]
    })
    submission_best.to_csv(f'ensemble_{best_method[0]}_submission.csv', index=False)
    print(f"ensemble_{best_method[0]}_submission.csv 파일 생성 완료!")

print("\n" + "="*70)
print("앙상블 완료!")
print("="*70)


LightGBM + XGBoost 앙상블

[1단계] OOF 예측값으로 앙상블 가중치 탐색
----------------------------------------------------------------------
가중치 탐색 중...
LGB 가중치: 0.0, XGB 가중치: 1.0 -> Gini: 0.277603
LGB 가중치: 0.1, XGB 가중치: 0.9 -> Gini: 0.278712
LGB 가중치: 0.2, XGB 가중치: 0.8 -> Gini: 0.279665
LGB 가중치: 0.3, XGB 가중치: 0.7 -> Gini: 0.280468
LGB 가중치: 0.4, XGB 가중치: 0.6 -> Gini: 0.281126
LGB 가중치: 0.5, XGB 가중치: 0.5 -> Gini: 0.281640
LGB 가중치: 0.6, XGB 가중치: 0.4 -> Gini: 0.282012
LGB 가중치: 0.7, XGB 가중치: 0.3 -> Gini: 0.282247
LGB 가중치: 0.8, XGB 가중치: 0.2 -> Gini: 0.282345
LGB 가중치: 0.9, XGB 가중치: 0.1 -> Gini: 0.282310
LGB 가중치: 1.0, XGB 가중치: 0.0 -> Gini: 0.282140

최적 가중치:
  LightGBM: 0.8
  XGBoost: 0.2
  앙상블 Gini: 0.282345

[2단계] 개별 모델 vs 앙상블 성능 비교
LightGBM Gini:     0.282140
XGBoost Gini:      0.277603
앙상블 Gini:       0.282345

개선도:
  vs LightGBM:  +0.000205
  vs XGBoost:   +0.004742

[3단계] 테스트 데이터 앙상블 예측
테스트 예측값 통계:
  Min:  0.008210
  Max:  0.242659
  Mean: 0.036379
  Std:  0.018923

ensemble_lgb_xgb_submission.csv 파일 생성 완료!

# 과제2 - 실습 및 과제 1.1~3에서 제외한 나만의 성능개선 방법을 시도해보고 이에 대해 설명 하라